# Model 3: Generative LLM (advanced/exploratory)
Weak Supervision

In this model, full text reviews are fed to a generative LLM which “reads” the reviews and returns aspect-sentiment pairs in a defined JSON format. The OpenAI model `gpt-4.1-mini` was selected because it offers strong performance on structured extraction tasks, reliable instruction following, and significantly lower cost and latency than larger models, making it practical for processing thousands of reviews via the API.

In this notebook, the models are tested on the data labelled for weak supervision. See `07b` for this model's performance on the manually labelled "gold standard" data.

## OpenAI API Key

This notebook requires an OpenAI API key.

When running the notebook, you will be prompted to enter your key.

You can obtain a key from:
https://platform.openai.com/api-keys

Approximate cost and time of each API call is reported in the markdown text before the call.

In [1]:
# Install/upgrade the OpenAI Python client
%pip install -U openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Securely request OpenAI API Key from notebook user
import os
from getpass import getpass
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Select OpenAI model and prompt lag time
MODEL = "gpt-4.1-mini"
SLEEP_SECONDS = 0.2

Enter your OpenAI API key:  ········


## Imports

In [3]:
import json
import time
import random
import warnings
import pandas as pd
from collections import defaultdict

from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

warnings.filterwarnings("ignore")

## A. Weak Supervision Performance

First, the model is analyzed using the data labeled for weak supervision. This data contains the full review text, as well as, the weakly-assigned ABSA labels. See the data processing (`01_process_data.ipynb`) and feature engineering (`03_feature_engineering.ipynb`) notebooks for more details about this labeling.


### A.1. Load Data

The cleaned, full-review, weakly-labeled dataset is read from the CSV source in chunks to avoid loading the entire dataset into memory.

A smaller subset of reviews is then selected for analysis due to the computational time and API cost associated with prompting a generative LLM.

The subset is intentionally constructed to be balanced across rating levels and to limit the influence of any single business by allowing no more than three reviews per business (`gmap_id`). Additionally, reviews that were selected for manual ABSA labelling are removed to prevent leakage.

In [4]:
# ---------------------------------------------------------------------------------------
# DATA LOADING SETTINGS
# ---------------------------------------------------------------------------------------

# Define raw data file & manually-labelled ABSA training data
INFILE = "../data/cleaned_for_LLM.csv"
ABSA_FILE = "../data/absa_training_set.csv"

# Select chunk size and sample size, also random state so that the same dataset is generated if the code is rerun
SAMPLE_SIZE = 2500
CHUNK_SIZE = 750000
RANDOM_STATE = 120

# Column labels
LABEL_COLS = [
    "product_quality_positive",
    "product_quality_negative",
    "service_positive",
    "service_negative",
    "wait_time_positive",
    "wait_time_negative",
    "price_value_positive",
    "price_value_negative",
    "cleanliness_positive",
    "cleanliness_negative",
    "atmosphere_positive",
    "atmosphere_negative",
    "general_positive",
    "general_negative",
]

USECOLS = ["review_id", "rating", "text", "gmap_id"] + LABEL_COLS

# Save paths
EVAL_SAMPLE_OUTFILE = "../data/openai_eval_sample.csv"
ZERO_RESULTS_OUTFILE = "../data/openai_zero_shot_results.csv"
FEW_RESULTS_OUTFILE = "../data/openai_few_shot_results.csv"
COMPARISON_OUTFILE = "../data/openai_comparison_metrics.csv"

In [5]:
# ------------------------------------------------------------------------
# DATA LOADING / SAMPLING FUNCTION
# ------------------------------------------------------------------------

def read_balanced_sample_by_rating_and_gmap(
    infile,
    absa_file,
    usecols,
    sample_size,
    chunk_size,
    random_state,
    allowed_ratings=(1, 2, 3, 4, 5),
    max_per_gmap_per_rating=3
):
    """
    Read a large CSV in chunks, exclude review_ids from the manual ABSA set,
    and build a subset that is:
      1) balanced across ratings
      2) spread across gmap_id values within each rating bucket
    """
    random.seed(random_state)

    allowed_ratings = list(allowed_ratings)
    n_ratings = len(allowed_ratings)

    target_per_rating = sample_size // n_ratings
    remainder = sample_size % n_ratings

    target_counts = {r: target_per_rating for r in allowed_ratings}
    for r in allowed_ratings[:remainder]:
        target_counts[r] += 1

    print("Target counts by rating:")
    for r in allowed_ratings:
        print(f"  Rating {r}: {target_counts[r]:,}")

    # Load manual ABSA review_ids to exclude
    absa_ids = set(
        pd.read_csv(absa_file, usecols=["review_id"])["review_id"].astype(str)
    )
    print(f"\nLoaded {len(absa_ids):,} ABSA review_ids to exclude")

    pools = {r: [] for r in allowed_ratings}
    gmap_counts = {r: defaultdict(int) for r in allowed_ratings}
    valid_seen = {r: 0 for r in allowed_ratings}

    reader = pd.read_csv(
        infile,
        usecols=usecols,
        chunksize=chunk_size
    )

    for i, chunk in enumerate(reader, start=1):
        chunk["review_id"] = chunk["review_id"].astype(str)
        chunk["gmap_id"] = chunk["gmap_id"].astype(str)

        # Filter out manually labeled reviews
        chunk = chunk[~chunk["review_id"].isin(absa_ids)]

        # Keep only selected ratings
        chunk = chunk[chunk["rating"].isin(allowed_ratings)]

        if len(chunk) == 0:
            print(f"Chunk {i}: 0 valid rows after filtering")
            continue

        # Shuffle within chunk
        chunk = chunk.sample(frac=1, random_state=random_state + i).reset_index(drop=True)

        for row in chunk.to_dict(orient="records"):
            r = row["rating"]
            g = row["gmap_id"]

            valid_seen[r] += 1

            if gmap_counts[r][g] < max_per_gmap_per_rating:
                pools[r].append(row)
                gmap_counts[r][g] += 1

        msg = [f"Chunk {i}"]
        for r in allowed_ratings:
            msg.append(
                f"r{r}: seen={valid_seen[r]:,}, kept={len(pools[r]):,}, gmaps={len(gmap_counts[r]):,}"
            )
        print(" | ".join(msg))

    final_parts = []

    print("\nFinal bucket sizes before downsampling:")
    for r in allowed_ratings:
        print(f"  Rating {r}: {len(pools[r]):,}")

    for r in allowed_ratings:
        bucket = pools[r]
        target = target_counts[r]

        if len(bucket) == 0:
            print(f"WARNING: Rating {r} has 0 rows after filtering")
            continue

        if len(bucket) >= target:
            bucket_df = pd.DataFrame(bucket).sample(n=target, random_state=random_state)
        else:
            print(
                f"WARNING: Rating {r} only has {len(bucket):,} rows available "
                f"(target was {target:,})"
            )
            bucket_df = pd.DataFrame(bucket)

        final_parts.append(bucket_df)

    df_sample = pd.concat(final_parts, ignore_index=True)
    df_sample = df_sample.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nDone.")
    print(f"Final sample size: {len(df_sample):,}\n")

    print("Final counts by rating:")
    print(df_sample["rating"].value_counts().sort_index())

    print("\nUnique gmap_id counts by rating:")
    print(df_sample.groupby("rating")["gmap_id"].nunique().sort_index())

    return df_sample

In [6]:
# ------------------------------------------------------------
# Build Subset & Save to .CSV
# ------------------------------------------------------------

df_sample = read_balanced_sample_by_rating_and_gmap(
    infile=INFILE,
    absa_file=ABSA_FILE,
    usecols=USECOLS,
    sample_size=SAMPLE_SIZE,
    chunk_size=CHUNK_SIZE,
    random_state=RANDOM_STATE,
    allowed_ratings=(1, 2, 3, 4, 5),
    max_per_gmap_per_rating=3
)

df_sample.to_csv(EVAL_SAMPLE_OUTFILE, index=False)
print(f"\nSaved evaluation sample to: {EVAL_SAMPLE_OUTFILE}")

display(df_sample.head(3))

Target counts by rating:
  Rating 1: 500
  Rating 2: 500
  Rating 3: 500
  Rating 4: 500
  Rating 5: 500

Loaded 10,000 ABSA review_ids to exclude
Chunk 1 | r1: seen=75,980, kept=49,320, gmaps=24,692 | r2: seen=23,444, kept=20,410, gmaps=14,057 | r3: seen=40,906, kept=31,863, gmaps=19,085 | r4: seen=94,003, kept=61,721, gmaps=29,688 | r5: seen=515,351, kept=117,717, gmaps=39,653
Chunk 2 | r1: seen=123,490, kept=67,743, gmaps=33,219 | r2: seen=49,963, kept=31,675, gmaps=20,136 | r3: seen=102,195, kept=48,303, gmaps=26,941 | r4: seen=237,325, kept=87,357, gmaps=40,343 | r5: seen=986,416, kept=154,989, gmaps=52,159
Chunk 3 | r1: seen=179,145, kept=98,796, gmaps=48,211 | r2: seen=74,289, kept=48,532, gmaps=30,405 | r3: seen=153,587, kept=75,978, gmaps=41,028 | r4: seen=359,531, kept=133,473, gmaps=59,607 | r5: seen=1,482,486, kept=222,400, gmaps=74,781
Chunk 4 | r1: seen=236,810, kept=133,224, gmaps=64,370 | r2: seen=99,253, kept=69,287, gmaps=42,291 | r3: seen=204,095, kept=109,375, gmaps

,review_id,rating,text,gmap_id,product_quality_positive,product_quality_negative,service_positive,service_negative,wait_time_positive,wait_time_negative,price_value_positive,price_value_negative,cleanliness_positive,cleanliness_negative,atmosphere_positive,atmosphere_negative,general_positive,general_negative
0,afdbdfe66773e5242de19fd53c67a9e8aa0d24059274a2...,4,I came here for the courts. If you’re like me ...,0x80dd4b24b87c588b:0x83f5a8bfcbd000f1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,4b54b991262ea8d8f1a24b8ca816052603f3b8542e8931...,1,Ordered a pizza online. First their website s...,0x80904035350740a9:0x58b08c1c4e3e4a98,0,1,0,1,0,1,0,0,0,0,0,0,0,1
2,94738d3d2b40ea7b4fe4187bc3b0deb9d5b677462255fd...,4,Great customer service,0x80dd28beb00c9d31:0x605c4392c9aa8baa,0,0,1,0,0,0,0,0,0,0,0,0,1,0


### A.2. Generative LLM Prompt Creation

In [7]:
# -----------------------------------------------------------
# Define Output JSON
# ------------------------------------------------------------

ASPECTS = [
    "Product Quality",
    "Service",
    "Wait Time",
    "Price/Value",
    "Cleanliness",
    "Atmosphere",
    "General"
]

SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

ABSA_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "absa_review_output",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "review_id": {"type": "string"},
                "pairs": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "aspect": {"type": "string", "enum": ASPECTS},
                            "sentiment": {"type": "string", "enum": SENTIMENTS},
                            "evidence": {"type": "string"}
                        },
                        "required": ["aspect", "sentiment", "evidence"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["review_id", "pairs"],
            "additionalProperties": False
        }
    }
}

In [8]:
# ----------------------------------------------------------------------------
# Define Prompt
# ----------------------------------------------------------------------------

DEVELOPER_PROMPT = """
You are an expert annotator for aspect-based sentiment analysis (ABSA).

Task:
Read one customer review and extract all aspect-sentiment pairs present.

Allowed aspects:
- Product Quality
- Service
- Wait Time
- Price/Value
- Cleanliness
- Atmosphere
- General

Allowed sentiment labels:
- positive
- negative
- neutral
- mixed

Rules:
1. Return only the requested JSON schema.
2. Include one entry per distinct aspect mentioned in the review.
3. Do not invent aspects not supported by the review.
4. Use General if there is an overall sentiment that does not clearly fit another aspect.
5. Keep evidence short and directly grounded in the review text.
6. If an aspect contains both positive and negative sentiment, use mixed.
7. If there is no clear aspect sentiment, return an empty list for pairs.
"""

In [9]:
# --------------------------------------------------------------------------
# FEW-SHOT EXAMPLES
# --------------------------------------------------------------------------

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "Review ID: ex1\nReview: The tacos were amazing and cheap, but the dining room was dirty."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex1",
            "pairs": [
                {"aspect": "Product Quality", "sentiment": "positive", "evidence": "tacos were amazing"},
                {"aspect": "Price/Value", "sentiment": "positive", "evidence": "cheap"},
                {"aspect": "Cleanliness", "sentiment": "negative", "evidence": "dining room was dirty"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex2\nReview: Our server was kind, but we waited 40 minutes for the food."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex2",
            "pairs": [
                {"aspect": "Service", "sentiment": "positive", "evidence": "server was kind"},
                {"aspect": "Wait Time", "sentiment": "negative", "evidence": "waited 40 minutes"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex3\nReview: Great place overall. I'll definitely come back."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex3",
            "pairs": [
                {"aspect": "General", "sentiment": "positive", "evidence": "Great place overall"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex4\nReview: The restaurant looked nice but the tables were sticky."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex4",
            "pairs": [
                {"aspect": "Atmosphere", "sentiment": "positive", "evidence": "restaurant looked nice"},
                {"aspect": "Cleanliness", "sentiment": "negative", "evidence": "tables were sticky"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex5\nReview: The burger was okay and the price was fair."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex5",
            "pairs": [
                {"aspect": "Product Quality", "sentiment": "neutral", "evidence": "burger was okay"},
                {"aspect": "Price/Value", "sentiment": "neutral", "evidence": "price was fair"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex6\nReview: The staff were friendly but also forgot our drinks several times."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex6",
            "pairs": [
                {"aspect": "Service", "sentiment": "mixed", "evidence": "staff were friendly but forgot our drinks"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex7\nReview: The line moved quickly and the food came out fast."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex7",
            "pairs": [
                {"aspect": "Wait Time", "sentiment": "positive", "evidence": "line moved quickly"}
            ]
        })
    }
]

### A.3. Build Prompts

In [10]:
# Function to build prompt
def build_messages(review_id, review_text, mode="zero_shot"):
    messages = [{"role": "developer", "content": DEVELOPER_PROMPT}]

    if mode == "few_shot":
        messages.extend(FEW_SHOT_EXAMPLES)

    messages.append({
        "role": "user",
        "content": f"Review ID: {review_id}\nReview: {review_text}"
    })

    return messages

In [11]:
# Function to call API
def call_absa_model(review_id, review_text, mode="zero_shot", max_retries=3):
    """
    Send one review to OpenAI and return parsed structured output.
    Also returns token usage for later cost analysis.
    """
    messages = build_messages(review_id, review_text, mode=mode)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                response_format=ABSA_SCHEMA,
                temperature=0
            )

            content = response.choices[0].message.content
            parsed = json.loads(content)
            usage = response.usage

            if "review_id" not in parsed:
                parsed["review_id"] = str(review_id)
            if "pairs" not in parsed or parsed["pairs"] is None:
                parsed["pairs"] = []

            return {
                "review_id": str(review_id),
                "success": True,
                "parsed": parsed,
                "error": None,
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
            }

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return {
                    "review_id": str(review_id),
                    "success": False,
                    "parsed": {"review_id": str(review_id), "pairs": []},
                    "error": str(e),
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                }

In [12]:
# Map API output to 14 ABSA labels

ASPECT_TO_PREFIX = {
    "Product Quality": "product_quality",
    "Service": "service",
    "Wait Time": "wait_time",
    "Price/Value": "price_value",
    "Cleanliness": "cleanliness",
    "Atmosphere": "atmosphere",
    "General": "general"
}

def pairs_to_binary_label_dict(pairs):
    """
    Convert ABSA pairs into the same 14 binary columns as the weak ABSA labels.
    """
    out = {col: 0 for col in LABEL_COLS}

    for pair in pairs:
        aspect = pair.get("aspect")
        sentiment = pair.get("sentiment")

        if aspect not in ASPECT_TO_PREFIX:
            continue

        prefix = ASPECT_TO_PREFIX[aspect]

        if sentiment == "positive":
            out[f"{prefix}_positive"] = 1
        elif sentiment == "negative":
            out[f"{prefix}_negative"] = 1
        elif sentiment == "mixed":
            out[f"{prefix}_positive"] = 1
            out[f"{prefix}_negative"] = 1
        elif sentiment == "neutral":
            pass

    return out

### A.4. Prompt Model via API

In [13]:
# ------------------------------------------------------------
# FUNCTION FOR MODEL PROMPTING VIA API
# ------------------------------------------------------------

def run_openai_absa(df_input, mode="zero_shot", sleep_seconds=0.2, save_every=100, outfile=None):
    """
    Run OpenAI ABSA extraction over a dataframe and return a dataframe
    with predicted binary label columns added.

    If outfile is provided, partial checkpoints are saved every `save_every` rows.
    """
    results = []

    for idx, (_, row) in enumerate(tqdm(df_input.iterrows(), total=len(df_input)), start=1):
        review_id = str(row["review_id"])
        review_text = str(row["text"])

        result = call_absa_model(
            review_id=review_id,
            review_text=review_text,
            mode=mode,
            max_retries=3
        )

        pred_dict = pairs_to_binary_label_dict(result["parsed"].get("pairs", []))

        out_row = {
            "review_id": review_id,
            "rating": row["rating"],
            "text": row["text"],
            "gmap_id": row["gmap_id"],
            "mode": mode,
            "success": result["success"],
            "error": result["error"],
            "raw_pairs": json.dumps(result["parsed"].get("pairs", [])),
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"],
            "total_tokens": result["total_tokens"],
        }

        for col in LABEL_COLS:
            out_row[f"true__{col}"] = int(row[col])

        for col in LABEL_COLS:
            out_row[f"pred__{col}"] = int(pred_dict[col])

        results.append(out_row)

        if outfile is not None and idx % save_every == 0:
            pd.DataFrame(results).to_csv(outfile, index=False)
            print(f"\nCheckpoint saved: {outfile} ({idx:,} rows processed)")

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    results_df = pd.DataFrame(results)

    if outfile is not None:
        results_df.to_csv(outfile, index=False)
        print(f"\nFinal saved: {outfile}")

    return results_df

#### A.5. Run Zero-Shot

Using the default parameters provided, this API call took approximately 75 minutes and resulted in 927,440 input tokens and 223,330 output tokens.

As of March 2026, the cost of this using `gpt-4.1-mini` is ~$0.75.

In [14]:
results_zero = run_openai_absa(
    df_input=df_sample,
    mode="zero_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=ZERO_RESULTS_OUTFILE
)

  4%|▍         | 100/2500 [03:07<1:05:25,  1.64s/it]


Checkpoint saved: ../data/openai_zero_shot_results.csv (100 rows processed)


  8%|▊         | 200/2500 [06:15<1:03:39,  1.66s/it]


Checkpoint saved: ../data/openai_zero_shot_results.csv (200 rows processed)


 12%|█▏        | 300/2500 [09:26<1:10:19,  1.92s/it]

...


Checkpoint saved: ../data/openai_zero_shot_results.csv (2,400 rows processed)


100%|██████████| 2500/2500 [1:15:09<00:00,  2.58s/it]


Checkpoint saved: ../data/openai_zero_shot_results.csv (2,500 rows processed)


100%|██████████| 2500/2500 [1:15:09<00:00,  1.80s/it]


Final saved: ../data/openai_zero_shot_results.csv


#### A.6. Run Few-Shot

Using the default parameters provided, this API call took approximately 70 minutes and resulted in 2,232,440 input tokens and 203,293 output tokens.

As of March 2026, the cost of this using `gpt-4.1-mini` is ~$1.25.

In [15]:
results_few = run_openai_absa(
    df_input=df_sample,
    mode="few_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=FEW_RESULTS_OUTFILE
)

  4%|▍         | 99/2500 [03:07<1:07:19,  1.68s/it]


Checkpoint saved: ../data/openai_few_shot_results.csv (100 rows processed)


  8%|▊         | 200/2500 [05:54<59:16,  1.55s/it]  


Checkpoint saved: ../data/openai_few_shot_results.csv (200 rows processed)


 12%|█▏        | 300/2500 [08:36<1:06:43,  1.82s/it]

...


Checkpoint saved: ../data/openai_few_shot_results.csv (2,400 rows processed)


100%|██████████| 2500/2500 [1:09:04<00:00,  2.18s/it]


Checkpoint saved: ../data/openai_few_shot_results.csv (2,500 rows processed)


100%|██████████| 2500/2500 [1:09:04<00:00,  1.66s/it]


Final saved: ../data/openai_few_shot_results.csv


### A.7. Weak Supervision Evaluation

In [16]:
# ------------------------------------------------------------
# Double check that all API calls were successfull
# -------------------------------------------------------------

def summarize_success(results_df, mode_name):
    n_total = len(results_df)
    n_success = results_df["success"].sum()
    n_failed = n_total - n_success

    print(f"\n{mode_name} API call summary")
    print(f"  total rows: {n_total:,}")
    print(f"  successful: {n_success:,}")
    print(f"  failed:     {n_failed:,}")

In [17]:
summarize_success(results_zero, "ZERO-SHOT")
summarize_success(results_few, "FEW-SHOT")


ZERO-SHOT API call summary
  total rows: 2,500
  successful: 2,500
  failed:     0

FEW-SHOT API call summary
  total rows: 2,500
  successful: 2,500
  failed:     0


In [18]:
# -----------------------------------------------------------
# Function for multi-label evaluation of model performance
# ------------------------------------------------------------

def evaluate_multilabel_results(results_df, label_cols):
    """
    Compute:
    - weighted precision
    - weighted recall
    - weighted F1
    - macro F1
    - subset accuracy
    - classification report
    """
    true_cols = [f"true__{c}" for c in label_cols]
    pred_cols = [f"pred__{c}" for c in label_cols]

    y_true = results_df[true_cols].values
    y_pred = results_df[pred_cols].values

    metrics = {
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "classification_report": classification_report(
            y_true,
            y_pred,
            target_names=label_cols,
            zero_division=0
        )
    }

    return metrics


def print_multilabel_results(model_name, metrics):
    """
    Print results in a format that matches your other model output.
    """
    print(f"{model_name}")
    print(f"Test Accuracy:     {metrics['accuracy']:.4f}\n")
    print(f"F1 Score (macro):    {metrics['macro_f1']:.4f}")
    print(f"F1 Score (weighted): {metrics['weighted_f1']:.4f}\n")
    print("Classification Report:")
    print(metrics["classification_report"])

In [19]:
zero_metrics = evaluate_multilabel_results(results_zero, LABEL_COLS)
few_metrics = evaluate_multilabel_results(results_few, LABEL_COLS)

print_multilabel_results("ZERO-SHOT METRICS", zero_metrics)
print()
print_multilabel_results("FEW-SHOT METRICS", few_metrics)

ZERO-SHOT METRICS
Test Accuracy:     0.2076

F1 Score (macro):    0.5184
F1 Score (weighted): 0.5332

Classification Report:
                          precision    recall  f1-score   support

product_quality_positive       0.39      0.84      0.54       338
product_quality_negative       0.63      0.55      0.59       587
        service_positive       0.56      0.91      0.70       335
        service_negative       0.70      0.81      0.75       555
      wait_time_positive       0.54      0.17      0.26       157
      wait_time_negative       0.76      0.53      0.62       358
    price_value_positive       0.56      0.71      0.63       154
    price_value_negative       0.61      0.65      0.63       299
    cleanliness_positive       0.63      0.77      0.70        83
    cleanliness_negative       0.53      0.52      0.52       124
     atmosphere_positive       0.24      0.53      0.34       103
     atmosphere_negative       0.24      0.21      0.22       156
        general_

### A.8. Comparison of Zero-Shot vs. Few-Shot

In [20]:
# ------------------------------------------------------------
# FUNCTION TO CONVERT METRICS DICT TO A CLEAN COMPARISON ROW
# ------------------------------------------------------------

def metrics_to_row(mode_name, metrics):
    return {
        "mode": mode_name,
        "accuracy": metrics["accuracy"],
        "precision_weighted": metrics["precision"],
        "recall_weighted": metrics["recall"],
        "f1_macro": metrics["macro_f1"],
        "f1_weighted": metrics["weighted_f1"],
    }

In [21]:
comparison = pd.DataFrame([
    metrics_to_row("zero_shot", zero_metrics),
    metrics_to_row("few_shot", few_metrics),
])

display(comparison)

comparison.to_csv(COMPARISON_OUTFILE, index=False)
print(f"\nSaved comparison metrics to: {COMPARISON_OUTFILE}")

,mode,accuracy,precision_weighted,recall_weighted,f1_macro,f1_weighted
0,zero_shot,0.2076,0.550641,0.557013,0.518406,0.533229
1,few_shot,0.2500,0.596952,0.519004,0.514822,0.529940



Saved comparison metrics to: ../data/openai_comparison_metrics.csv


## A.9. Token Usage Summary

In [22]:
# -----------------------------------------------------------
# Function to calculate token usage
# ------------------------------------------------------------

def summarize_usage(results_df, mode_name):
    prompt_tokens = pd.to_numeric(results_df["prompt_tokens"], errors="coerce").fillna(0).sum()
    completion_tokens = pd.to_numeric(results_df["completion_tokens"], errors="coerce").fillna(0).sum()
    total_tokens = pd.to_numeric(results_df["total_tokens"], errors="coerce").fillna(0).sum()

    print(f"\n{mode_name} token usage")
    print(f"  prompt_tokens:     {int(prompt_tokens):,}")
    print(f"  completion_tokens: {int(completion_tokens):,}")
    print(f"  total_tokens:      {int(total_tokens):,}")

summarize_usage(results_zero, "ZERO-SHOT")
summarize_usage(results_few, "FEW-SHOT")


ZERO-SHOT token usage
  prompt_tokens:     927,440
  completion_tokens: 223,330
  total_tokens:      1,150,770

FEW-SHOT token usage
  prompt_tokens:     2,232,440
  completion_tokens: 203,293
  total_tokens:      2,435,733
